# Semana 3: EDA Avanzado, Estacionariedad y Pregunta de Investigación

## Proyecto: Serie de Tiempo de Caudal – Estación Pueblo Nuevo (IDEAM)

**Objetivo:** Realizar un análisis exploratorio profundo de la serie limpia para identificar patrones temporales, evaluar estacionariedad y formular la pregunta de investigación que guiará el modelado predictivo.

### Contenido:
1. Importar librerías
2. Cargar serie limpia (exportada en Semana 2)
3. Descomposición estacional (STL: Trend + Seasonal + Residual)
4. Visualización de componentes descompuestos
5. Autocorrelación (ACF) y Autocorrelación Parcial (PACF)
6. Pruebas de estacionariedad (ADF y KPSS)
7. Pregunta de investigación
8. Análisis de estacionalidad mensual y trimestral
9. Patrones interanuales y tendencia
10. Distribución por épocas hidrológicas (seca vs lluviosa)
11. Resumen del EDA y hallazgos clave
12. Conclusiones y próximos pasos

## 1. Importar Librerías

In [1]:

# =============================================================================
# CONCEPTO: Importación de librerías para EDA avanzado de series de tiempo
# -----------------------------------------------------------------------------
# pandas / numpy        → manipulación de datos y cálculos numéricos
# plotly.express        → gráficos interactivos de alto nivel
# plotly.graph_objects  → trazas personalizadas (Scatter, Bar, etc.)
# make_subplots         → composición de figuras con múltiples ejes
#
# statsmodels.tsa.seasonal.STL
#   → descomposición estacional robusta: Trend + Seasonal + Residual
#   → "robust=True" usa regresión LOESS ponderada, reduce influencia de outliers
#
# statsmodels.tsa.stattools
#   - acf()      → Función de Autocorrelación: correlación Yt vs Yt-k
#   - pacf()     → Autocorrelación Parcial: correlación directa sin efectos intermedios
#   - adfuller() → Prueba ADF de raíz unitaria (H₀: no estacionaria)
#   - kpss()     → Prueba KPSS (H₀: estacionaria); complementaria a ADF
#
# statsmodels.graphics.tsaplots
#   - plot_acf / plot_pacf → gráficos estándar de correlogramas (matplotlib)
#     (importados aquí aunque se usen versiones Plotly en el análisis)
#
# warnings.filterwarnings("ignore")
#   → suprime advertencias de KPSS sobre truncamiento de lags
#
# pio.templates.default = "plotly_white"
#   → plantilla global de Plotly: fondo blanco, cuadrícula clara
#
# CRITERIO DE USO: Serie de tiempo con posible estacionalidad anual y tendencia.
# Se necesitan pruebas formales (ADF/KPSS) y herramientas de descomposición (STL)
# para caracterizar la serie antes del modelado predictivo.
# =============================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Análisis de series de tiempo
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import warnings
warnings.filterwarnings("ignore")

import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Librerías importadas")


✅ Librerías importadas


## 2. Cargar Serie Limpia (Exportada en Semana 2)

Usamos el CSV `caudal_limpio_diario.csv` generado en la Semana 2, que contiene la serie diaria completa con imputacion por climatologia estacional y capping aplicados.

In [2]:

# =============================================================================
# CONCEPTO: Carga de la serie limpia generada en la Semana 2
# -----------------------------------------------------------------------------
# pd.read_csv(ruta, index_col="Fecha", parse_dates=True)
#   - ruta relativa "../Week_2/..."  → portabilidad entre entornos
#   - index_col="Fecha"             → la columna Fecha se convierte en índice
#   - parse_dates=True              → el índice queda como DatetimeIndex (dtype datetime64)
#
# Verificaciones clave al cargar una serie de tiempo limpia:
#   - df.index.inferred_freq → confirma que la frecuencia es "D" (diaria)
#     Sin frecuencia definida, STL y los modelos ARIMA fallarían
#   - df["Caudal"].isna().sum() → debe ser 0 tras la imputación de Semana 2
#   - df.index.min() / max() → revisión del período completo disponible
#
# CRITERIO DE USO: Reutilizar series ya limpias evita replicar la pipeline
# de preprocesamiento. El archivo CSV exportado en la etapa anterior actúa
# como contrato de datos entre semanas.
# =============================================================================
# Cargar serie limpia
df = pd.read_csv("../Week_2/caudal_limpio_diario.csv", index_col="Fecha", parse_dates=True)

print(f"✅ Serie limpia cargada")
print(f"   Período:  {df.index.min().date()} → {df.index.max().date()}")
print(f"   Registros: {len(df)}")
print(f"   Columnas:  {list(df.columns)}")
print(f"   NaN:        {df['Caudal'].isna().sum()}")
print(f"   Frecuencia: {df.index.inferred_freq}")
print(f"\n📊 Estadísticas rápidas:")
df["Caudal"].describe()


✅ Serie limpia cargada
   Período:  2010-01-01 → 2017-12-31
   Registros: 2922
   Columnas:  ['Caudal', 'Zscore', 'Caudal_sin_cap', 'Caudal_log', 'Caudal_boxcox', 'Caudal_diff']
   NaN:        0
   Frecuencia: D

📊 Estadísticas rápidas:


count    2922.000000
mean        3.479962
std         1.626595
min         0.120804
25%         2.694474
50%         3.369384
75%         4.109110
max        12.475000
Name: Caudal, dtype: float64

## 3. Descomposición Estacional (STL)

La descomposición **STL** (*Seasonal and Trend decomposition using Loess*) separa la serie en tres componentes:

- **Tendencia (Trend):** Movimiento de largo plazo de la serie
- **Estacionalidad (Seasonal):** Patrón cíclico que se repite cada año (período = 365 días)
- **Residuo (Residual):** Variación no explicada por tendencia ni estacionalidad

$$Y_t = T_t + S_t + R_t$$

In [3]:

# =============================================================================
# CONCEPTO: Descomposición STL (Seasonal-Trend decomposition using Loess)
# -----------------------------------------------------------------------------
# STL(serie, period, robust)
#   - serie    → pd.Series con DatetimeIndex y frecuencia uniforme
#   - period   → longitud del ciclo estacional en unidades de la serie
#                365 días = estacionalidad anual sobre datos diarios
#   - robust=True → usa pesos bisquadrados en LOESS para reducir el impacto
#                   de outliers al estimar tendencia y estacionalidad
#
# stl.fit()   → ejecuta el algoritmo iterativo; devuelve objeto DecomposeResult
#   .trend    → componente de tendencia (suavizado LOESS de baja frecuencia)
#   .seasonal → componente estacional periódico (se repite cada 365 días)
#   .resid    → residuo = serie - tendencia - estacionalidad
#
# Métricas de fuerza (Wang et al., 2006):
#   Fuerza de tendencia      = 1 - Var(R) / Var(T + R)
#   Fuerza de estacionalidad = 1 - Var(R) / Var(S + R)
#   Rango [0, 1]: valores > 0.6 indican componente significativo
#
# STL vs. Descomposición clásica (seasonal_decompose):
#   ✅ STL maneja series con longitud no múltiplo del período
#   ✅ Robusto a outliers con robust=True
#   ✅ Permite que la amplitud estacional varíe a lo largo del tiempo
#
# CRITERIO DE USO: Paso previo al modelado SARIMA/Holt-Winters/Prophet.
# Cuantifica cuánta varianza explica la estacionalidad y la tendencia,
# orientando la elección de parámetros del modelo (d, D, m).
# =============================================================================
# Descomposición STL (período = 365 días para estacionalidad anual)
stl = STL(df["Caudal"], period=365, robust=True)
resultado = stl.fit()

print("✅ Descomposición STL completada")
print(f"   Período estacional: 365 días")
print(f"   Fuerza de tendencia:      {1 - resultado.resid.var() / (resultado.trend + resultado.resid).var():.3f}")
print(f"   Fuerza de estacionalidad: {1 - resultado.resid.var() / (resultado.seasonal + resultado.resid).var():.3f}")


✅ Descomposición STL completada
   Período estacional: 365 días
   Fuerza de tendencia:      0.303
   Fuerza de estacionalidad: -0.047


## 4. Visualización de Componentes Descompuestos

Cada componente se grafica por separado para entender su contribución a la serie original.

In [4]:

# =============================================================================
# CONCEPTO: Visualización de los 4 componentes STL con subplots compartidos
# -----------------------------------------------------------------------------
# make_subplots(rows=4, cols=1, shared_xaxes=True)
#   - 4 paneles verticales para cada componente de la descomposición
#   - shared_xaxes=True → zoom/pan sincronizado entre todos los paneles
#   - vertical_spacing   → separación relativa entre subplots (0–1)
#   - subplot_titles     → etiquetas sobre cada panel (lista de strings)
#
# go.Scatter(x, y, mode="lines", line=dict(color, width))
#   - modo "lines" sin marcadores: adecuado para series densas (miles de puntos)
#   - width=0.8 para la serie original (muchos puntos → trazo fino legible)
#
# Interpretación visual de los componentes:
#   Serie Original → variabilidad total compuesta
#   Tendencia      → dirección de largo plazo (¿crece/decrece el caudal?)
#   Estacionalidad → patrón repetitivo anual (picos en época lluviosa)
#   Residuo        → ruido y eventos atípicos no capturados por T + S
#
# fig.update_yaxes(title_text, row, col)
#   → etiqueta de eje Y por panel individual
#
# CRITERIO DE USO: Visualizar los 4 componentes por separado permite evaluar
# si la descomposición es correcta: la tendencia debe ser suave, la estacionalidad
# periódica y el residuo sin estructura sistemática (ruido blanco idealmente).
# =============================================================================
# Visualización de los 4 componentes STL
componentes = {
    "Serie Original": df["Caudal"],
    "Tendencia": resultado.trend,
    "Estacionalidad": resultado.seasonal,
    "Residuo": resultado.resid,
}

colores = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=list(componentes.keys()),
                    vertical_spacing=0.05)

for i, (nombre, serie) in enumerate(componentes.items(), 1):
    fig.add_trace(go.Scatter(
        x=serie.index, y=serie.values,
        mode="lines", name=nombre,
        line=dict(color=colores[i-1], width=0.8),
    ), row=i, col=1)

fig.update_layout(
    title="Descomposición STL de la Serie de Caudal (período = 365 días)",
    height=800, width=1000, showlegend=False,
)
fig.update_yaxes(title_text="m³/s", row=1, col=1)
fig.update_yaxes(title_text="m³/s", row=2, col=1)
fig.update_yaxes(title_text="m³/s", row=3, col=1)
fig.update_yaxes(title_text="m³/s", row=4, col=1)
fig.show()


## 5. Autocorrelación (ACF) y Autocorrelación Parcial (PACF)

- **ACF** (*Autocorrelation Function*): mide la correlación entre $Y_t$ y $Y_{t-k}$ para distintos lags $k$. Picos periódicos indican estacionalidad.
- **PACF** (*Partial ACF*): mide la correlación directa entre $Y_t$ y $Y_{t-k}$ eliminando efectos intermedios. Ayuda a determinar el orden $p$ de modelos AR.

Analizamos hasta **730 lags** (2 años) para captar el ciclo anual completo.

In [5]:

# =============================================================================
# CONCEPTO: Función de Autocorrelación (ACF) y Autocorrelación Parcial (PACF)
# -----------------------------------------------------------------------------
# acf(serie, nlags, fft=True)
#   - nlags   → número de lags a calcular (730 = 2 años para captar ciclo anual)
#   - fft=True → usa Transformada de Fourier para acelerar el cálculo en series largas
#   - Devuelve array de coeficientes de correlación ρ(k) para k=0,1,...,nlags
#
# pacf(serie, nlags)
#   - Se limita a 60 lags: la PACF pierde interpretabilidad a lags muy largos
#   - Método por defecto: "yw" (Yule-Walker); alternativa: "ols"
#
# Intervalo de confianza 95% (aproximación de Bartlett):
#   IC₉₅ = ±1.96 / √N   donde N = número de observaciones
#   Coeficientes fuera del IC son estadísticamente significativos
#
# Coloración dinámica en go.Bar:
#   Rojo (#d62728) si |valor| > IC_95 → lag significativo
#   Azul claro (#aec7e8) si no supera el umbral
#
# add_vline(x=365/730) → marca visual de los ciclos anuales
#   Pico en lag 365: confirma estacionalidad anual fuerte
#   Pico en lag 730: confirma que el patrón se repite dos veces (ciclo bi-anual)
#
# Lectura para modelado SARIMA:
#   ACF: decaimiento lento → AR o diferenciación necesaria
#   PACF: corte abrupto en lag p → orden autorregresivo AR(p)
#   Pico estacional en ACF lag 365 → parámetro SAR/SMA estacional (m=365)
#
# CRITERIO DE USO: ACF + PACF son las herramientas diagnósticas estándar
# para identificar los órdenes (p, d, q)(P, D, Q)m de modelos SARIMA.
# =============================================================================
# Calcular ACF y PACF
n_lags = 730
acf_vals = acf(df["Caudal"], nlags=n_lags, fft=True)
pacf_vals = pacf(df["Caudal"], nlags=60)  # PACF solo para lags cortos

# Intervalo de confianza 95%
ic_95 = 1.96 / np.sqrt(len(df))

# ACF
fig_acf = go.Figure()
fig_acf.add_trace(go.Bar(
    x=list(range(n_lags + 1)), y=acf_vals,
    marker_color=["#d62728" if abs(v) > ic_95 else "#aec7e8" for v in acf_vals],
    name="ACF",
))
fig_acf.add_hline(y=ic_95, line_dash="dash", line_color="red", annotation_text="IC 95%")
fig_acf.add_hline(y=-ic_95, line_dash="dash", line_color="red")
fig_acf.add_hline(y=0, line_color="black", line_width=0.5)

# Marcar lags anuales
for yr in [365, 730]:
    fig_acf.add_vline(x=yr, line_dash="dot", line_color="green",
                      annotation_text=f"Lag {yr}")

fig_acf.update_layout(
    title=f"Función de Autocorrelación (ACF) – {n_lags} lags",
    xaxis_title="Lag (días)", yaxis_title="Autocorrelación",
    width=1000, height=400,
)
fig_acf.show()

# PACF (primeros 60 lags)
fig_pacf = go.Figure()
fig_pacf.add_trace(go.Bar(
    x=list(range(len(pacf_vals))), y=pacf_vals,
    marker_color=["#d62728" if abs(v) > ic_95 else "#aec7e8" for v in pacf_vals],
    name="PACF",
))
fig_pacf.add_hline(y=ic_95, line_dash="dash", line_color="red", annotation_text="IC 95%")
fig_pacf.add_hline(y=-ic_95, line_dash="dash", line_color="red")
fig_pacf.add_hline(y=0, line_color="black", line_width=0.5)
fig_pacf.update_layout(
    title="Función de Autocorrelación Parcial (PACF) – 60 lags",
    xaxis_title="Lag (días)", yaxis_title="PACF",
    width=1000, height=400,
)
fig_pacf.show()

print(f"📌 ACF en lag 1: {acf_vals[1]:.4f}")
print(f"📌 ACF en lag 365: {acf_vals[365]:.4f}")
print(f"📌 ACF en lag 730: {acf_vals[730]:.4f}")
print(f"📌 PACF en lag 1: {pacf_vals[1]:.4f}")


📌 ACF en lag 1: 0.9932
📌 ACF en lag 365: 0.0398
📌 ACF en lag 730: -0.1130
📌 PACF en lag 1: 0.9935


## 6. Pruebas de Estacionariedad (ADF y KPSS)

Una serie es **estacionaria** si sus propiedades estadísticas (media, varianza) no cambian con el tiempo.  
Esto es requisito para modelos ARIMA/SARIMA.

| Prueba | $H_0$ (Hipótesis nula) | Criterio |
|--------|----------------------|----------|
| **ADF** (Augmented Dickey-Fuller) | La serie tiene raíz unitaria (NO estacionaria) | Rechazar si $p < 0.05$ → estacionaria |
| **KPSS** (Kwiatkowski–Phillips–Schmidt–Shin) | La serie ES estacionaria | Rechazar si $p < 0.05$ → NO estacionaria |

Aplicamos ambas pruebas sobre la serie **original** y la **diferenciada**.

In [6]:

# =============================================================================
# CONCEPTO: Pruebas formales de estacionariedad — ADF y KPSS
# -----------------------------------------------------------------------------
# Una serie ESTACIONARIA tiene media, varianza y autocovarianza constantes en
# el tiempo. Los modelos ARIMA/SARIMA requieren estacionariedad.
#
# adfuller(serie, autolag="AIC")
#   H₀: la serie tiene raíz unitaria → NO es estacionaria
#   H₁: la serie no tiene raíz unitaria → ES estacionaria
#   autolag="AIC" → determina el número de lags del modelo aumentado
#                   minimizando el criterio de información de Akaike
#   Devuelve: (estadístico, p-valor, lags, n_obs, valores_críticos, ic_info)
#   Criterio: rechazar H₀ si p-valor < 0.05 → serie estacionaria
#
# kpss(serie, regression="c", nlags="auto")
#   H₀: la serie ES estacionaria (alrededor de una constante)
#   H₁: la serie NO es estacionaria (tiene raíz unitaria o tendencia)
#   regression="c" → prueba estacionariedad en niveles (sin tendencia)
#   regression="ct" → incluye tendencia lineal en H₀
#   nlags="auto"   → selección automática de lags por método de Newey-West
#   Criterio: rechazar H₀ si p-valor < 0.05 → serie NO estacionaria
#
# Combinación ADF + KPSS (tabla de diagnóstico):
#   ADF rechaza + KPSS NO rechaza → estacionaria (consenso)
#   ADF NO rechaza + KPSS rechaza → no estacionaria (consenso)
#   Ambas rechazan → estacionaria en tendencia, no en nivel (KPSS detecta tendencia)
#   Ninguna rechaza → evidencia insuficiente (serie fraccionalmente integrada)
#
# Serie diferenciada df["Caudal_diff"] = df["Caudal"].diff()
#   diff() orden 1 → elimina raíz unitaria simple I(1)
#   Si la serie original es I(1), la diferenciada será I(0) → estacionaria
#   → parámetro d=1 en ARIMA(p,d,q)
#
# CRITERIO DE USO: Aplicar ambas pruebas (ADF + KPSS) siempre en conjunto,
# ya que son complementarias y sus hipótesis nulas son opuestas, lo que
# permite detectar casos ambiguos con mayor confianza.
# =============================================================================
# Función para aplicar pruebas ADF y KPSS
def pruebas_estacionariedad(serie, nombre="Serie"):
    print(f"\n{'='*60}")
    print(f"📊 PRUEBAS DE ESTACIONARIEDAD: {nombre}")
    print(f"{'='*60}")

    # ADF
    adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(serie.dropna(), autolag="AIC")
    print(f"\n🔹 Prueba ADF (Augmented Dickey-Fuller):")
    print(f"   Estadístico: {adf_stat:.4f}")
    print(f"   p-valor:     {adf_p:.6f}")
    print(f"   Lags usados: {adf_lags}")
    print(f"   Valores críticos:")
    for key, val in adf_crit.items():
        print(f"     {key}: {val:.4f}")
    adf_result = "✅ ESTACIONARIA" if adf_p < 0.05 else "❌ NO estacionaria"
    print(f"   Conclusión ADF: {adf_result}")

    # KPSS
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(serie.dropna(), regression="c", nlags="auto")
    print(f"\n🔹 Prueba KPSS:")
    print(f"   Estadístico: {kpss_stat:.4f}")
    print(f"   p-valor:     {kpss_p:.4f}")
    print(f"   Lags usados: {kpss_lags}")
    print(f"   Valores críticos:")
    for key, val in kpss_crit.items():
        print(f"     {key}: {val:.4f}")
    kpss_result = "✅ ESTACIONARIA" if kpss_p >= 0.05 else "❌ NO estacionaria"
    print(f"   Conclusión KPSS: {kpss_result}")

    return adf_p, kpss_p

# Serie original
adf_p_orig, kpss_p_orig = pruebas_estacionariedad(df["Caudal"], "Caudal Original")

# Serie diferenciada (orden 1)
df["Caudal_diff"] = df["Caudal"].diff()
adf_p_diff, kpss_p_diff = pruebas_estacionariedad(df["Caudal_diff"].dropna(), "Caudal Diferenciado (d=1)")



📊 PRUEBAS DE ESTACIONARIEDAD: Caudal Original

🔹 Prueba ADF (Augmented Dickey-Fuller):
   Estadístico: -3.6546
   p-valor:     0.004799
   Lags usados: 28
   Valores críticos:
     1%: -3.4326
     5%: -2.8625
     10%: -2.5673
   Conclusión ADF: ✅ ESTACIONARIA

🔹 Prueba KPSS:
   Estadístico: 1.2366
   p-valor:     0.0100
   Lags usados: 31
   Valores críticos:
     10%: 0.3470
     5%: 0.4630
     2.5%: 0.5740
     1%: 0.7390
   Conclusión KPSS: ❌ NO estacionaria

📊 PRUEBAS DE ESTACIONARIEDAD: Caudal Diferenciado (d=1)

🔹 Prueba ADF (Augmented Dickey-Fuller):
   Estadístico: -12.8948
   p-valor:     0.000000
   Lags usados: 27
   Valores críticos:
     1%: -3.4326
     5%: -2.8625
     10%: -2.5673
   Conclusión ADF: ✅ ESTACIONARIA

🔹 Prueba KPSS:
   Estadístico: 0.0294
   p-valor:     0.1000
   Lags usados: 23
   Valores críticos:
     10%: 0.3470
     5%: 0.4630
     2.5%: 0.5740
     1%: 0.7390
   Conclusión KPSS: ✅ ESTACIONARIA


In [7]:

# =============================================================================
# CONCEPTO: Verificación visual de estacionariedad — media y varianza móvil
# -----------------------------------------------------------------------------
# Una serie estacionaria tiene media y varianza constantes en el tiempo.
# Graficar las estadísticas móviles permite detectar violaciones de este
# supuesto antes de aplicar modelos (complemento visual a ADF/KPSS).
#
# .rolling(window=365).mean()
#   → media aritmética calculada sobre los últimos 365 días (ventana deslizante)
#   → si la media móvil es aproximadamente horizontal → media constante (estacionariedad)
#   → si sube/baja sistemáticamente → tendencia presente (no estacionaria)
#
# .rolling(window=365).std()
#   → desviación estándar sobre la ventana deslizante
#   → si es aproximadamente constante → homocedasticidad (varianza constante)
#   → si varía con el tiempo → heterocedasticidad (puede requerir log-transformación)
#
# window=365 → ventana anual: suficientemente larga para suavizar el ciclo
#   estacional y revelar la tendencia subyacente, sin distorsionar por ruido puntual
#
# Subplots con shared_xaxes=True:
#   → panel 1: serie original + media móvil superpuesta
#   → panel 2: desviación estándar móvil (solo la estadística, sin la serie)
#   Comparar la línea roja (media móvil) con una línea horizontal ideal
#
# CRITERIO DE USO: Siempre graficar media y std móvil antes de aplicar
# pruebas formales. Si la media móvil presenta ciclos o tendencias y la
# std varía sistemáticamente, la serie requiere diferenciación o transformación
# antes del modelado.
# =============================================================================
# Visualización: media y varianza móvil para verificar estacionariedad visualmente
window = 365

rolling_mean = df["Caudal"].rolling(window=window).mean()
rolling_std = df["Caudal"].rolling(window=window).std()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Media Móvil (ventana = 365 días)",
                                    "Desviación Estándar Móvil (ventana = 365 días)"],
                    vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=df.index, y=df["Caudal"], mode="lines",
              name="Caudal", line=dict(color="#aec7e8", width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling_mean.index, y=rolling_mean, mode="lines",
              name="Media móvil", line=dict(color="#d62728", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling_std.index, y=rolling_std, mode="lines",
              name="Std móvil", line=dict(color="#ff7f0e", width=2)), row=2, col=1)

fig.update_layout(title="Verificación Visual de Estacionariedad",
                  width=1000, height=550, hovermode="x unified")
fig.update_yaxes(title_text="m³/s", row=1, col=1)
fig.update_yaxes(title_text="m³/s", row=2, col=1)
fig.show()


## 7. Pregunta de Investigación

Con base en los hallazgos del EDA (Semanas 1-3), formulamos la pregunta que guiará el modelado predictivo:

---

### **¿Es posible pronosticar el caudal medio diario de la estación Pueblo Nuevo con un horizonte de 30 días, utilizando modelos de series temporales (SARIMA, Holt-Winters, Prophet) y aprovechando la estacionalidad anual identificada?**

---

**Sub-preguntas:**
1. ¿Qué tan fuerte es la estacionalidad anual y cómo afecta la precisión del pronóstico?
2. ¿Cuál de los tres modelos (SARIMA, Holt-Winters, Prophet) ofrece mejor desempeño predictivo (RMSE, MAE, MAPE)?
3. ¿La transformación logarítmica mejora la capacidad predictiva al estabilizar la varianza?

**Variable objetivo:** Caudal medio diario (m³/s)  
**Horizonte de pronóstico:** 30 días  
**Métrica principal:** RMSE (Root Mean Squared Error)

## 8. Análisis de Estacionalidad Mensual y Trimestral

Profundizamos en los patrones cíclicos del caudal agrupando por **mes** y por **trimestre** para cuantificar la variabilidad estacional.

In [8]:

# =============================================================================
# CONCEPTO: Extracción de variables temporales y distribución mensual (Violin)
# -----------------------------------------------------------------------------
# Variables temporales derivadas del DatetimeIndex:
#   df.index.month   → entero 1–12 (número de mes)
#   df.index.year    → entero (año)
#   df.index.quarter → entero 1–4 (trimestre)
#
# .map(dict(enumerate(meses, 1)))
#   → convierte el número de mes en su nombre abreviado en español
#   → enumerate(meses, 1): genera pares (1,"Ene"), (2,"Feb"), ...
#   → dict() lo convierte en diccionario {1:"Ene", 2:"Feb", ...}
#   → usado con category_orders para forzar el orden correcto en el gráfico
#
# px.violin(df, x="Mes_nombre", y="Caudal", box=True, points=False)
#   - Violin plot: muestra la distribución completa (densidad de kernel)
#   - box=True       → incrusta un boxplot dentro del violín
#   - points=False   → no muestra puntos individuales (serie muy densa)
#   - color="Mes_nombre" + color_discrete_sequence → color único por mes
#   - category_orders → fuerza el orden cronológico Ene…Dic en eje X
#
# ¿Por qué violin y no solo boxplot?
#   El violin muestra la forma completa de la distribución (bimodalidad,
#   asimetría, multimodalidad) que el boxplot oculta con solo 5 estadísticos.
#
# CRITERIO DE USO: Usar violin plots para comparar distribuciones mensuales
# cuando se sospecha que la forma varía entre grupos (época seca vs lluviosa).
# Identificar los meses de picos y valles orienta la selección de m en SARIMA.
# =============================================================================
# Variables temporales
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
df["Mes"] = df.index.month
df["Año"] = df.index.year
df["Mes_nombre"] = df["Mes"].map(dict(enumerate(meses, 1)))
df["Trimestre"] = df.index.quarter
df["Trimestre_label"] = "Q" + df["Trimestre"].astype(str)

# Violin plot mensual
fig = px.violin(
    df.reset_index(), x="Mes_nombre", y="Caudal",
    box=True, points=False,
    title="Distribución del Caudal por Mes (Violin Plot)",
    labels={"Mes_nombre": "", "Caudal": "Caudal (m³/s)"},
    color="Mes_nombre",
    color_discrete_sequence=px.colors.qualitative.Set3,
    category_orders={"Mes_nombre": meses},
)
fig.update_layout(width=1000, height=500, showlegend=False)
fig.show()


In [9]:

# =============================================================================
# CONCEPTO: Boxplot trimestral y tabla de resumen estadístico mensual
# -----------------------------------------------------------------------------
# px.box(df, x="Trimestre_label", y="Caudal", color="Trimestre_label")
#   - Boxplot por trimestre: Q1 (Ene–Mar), Q2 (Abr–Jun), Q3 (Jul–Sep), Q4 (Oct–Dic)
#   - Agrupa múltiples años en cada caja → muestra variabilidad inter e intra-trimestral
#   - category_orders → fuerza el orden Q1→Q4 independientemente del orden en el DataFrame
#   - showlegend=False → el color ya está codificado en el eje X
#
# ¿Cuándo usar boxplot trimestral?
#   Cuando el régimen hidrológico sigue un patrón bimodal (Colombia) y se quiere
#   comparar la variabilidad a escala de 3 meses para validar la estacionalidad.
#
# df.groupby("Mes_nombre")["Caudal"].agg(["mean","std","median","min","max"])
#   → agrupación por nombre de mes → tabla de estadísticos descriptivos
#   .round(2) → redondea para presentación
#   .reindex(meses) → reordena las filas en orden cronológico (sin esto, el
#                     orden sería alfabético: Abr, Ago, Dic, ...)
#   .columns = [...]  → renombra las columnas en español
#
# display() → función de IPython/Jupyter que renderiza DataFrames con formato HTML
#   Preferible a print() para tablas en notebooks (permite scroll y formato)
#
# CRITERIO DE USO: La tabla de resumen mensual es un entregable clave del EDA:
# identifica los meses de mayor caudal (periodo lluvioso) y los meses de estiaje,
# cuantifica la amplitud de la estacionalidad (max/min) y orienta la definición
# de épocas hidrológicas para análisis posteriores.
# =============================================================================
# Boxplot trimestral
fig = px.box(
    df.reset_index(), x="Trimestre_label", y="Caudal",
    color="Trimestre_label",
    title="Distribución del Caudal por Trimestre",
    labels={"Trimestre_label": "Trimestre", "Caudal": "Caudal (m³/s)"},
    color_discrete_sequence=px.colors.qualitative.Pastel,
    category_orders={"Trimestre_label": ["Q1", "Q2", "Q3", "Q4"]},
)
fig.update_layout(width=700, height=450, showlegend=False)
fig.show()

# Tabla resumen mensual
resumen_mensual = df.groupby("Mes_nombre")["Caudal"].agg(["mean", "std", "median", "min", "max"]).round(2)
resumen_mensual = resumen_mensual.reindex(meses)
resumen_mensual.columns = ["Media", "Desv.Std", "Mediana", "Mínimo", "Máximo"]
print("📊 Resumen estadístico por mes:")
display(resumen_mensual)


📊 Resumen estadístico por mes:


,Media,Desv.Std,Mediana,Mínimo,Máximo
Mes_nombre,,,,,
Ene,3.12,1.51,3.49,0.14,6.48
Feb,2.89,1.46,3.03,0.13,8.34
Mar,3.04,1.46,3.32,0.12,6.84
Abr,3.11,1.40,3.32,0.24,7.70
May,3.26,1.26,3.36,0.77,5.68
Jun,3.74,1.69,3.47,0.73,12.48
Jul,3.93,1.34,3.63,1.39,8.52
Ago,3.81,1.55,3.33,1.44,8.15
Sep,3.50,1.39,3.24,1.27,7.13


## 9. Patrones Interanuales y Tendencia

¿Hay años significativamente más secos o más húmedos? Comparamos el caudal promedio anual y la evolución temporal año a año.

In [10]:

# =============================================================================
# CONCEPTO: Análisis de variabilidad interanual — caudal promedio por año
# -----------------------------------------------------------------------------
# df.groupby("Año")["Caudal"].agg(["mean","std","median"])
#   → agrupación por año: cada fila representa un año completo
#   → "mean"   → caudal promedio anual (indicador del año seco/húmedo)
#   → "std"    → dispersión dentro del año (relación con la estacionalidad)
#   → "median" → robusto a valores extremos puntuales
#
# px.bar(anual, x="Año", y="Media", error_y="Desv_Std")
#   - error_y="Desv_Std" → barras de error ±1σ sobre cada barra
#   - text=anual["Media"].round(1) → etiqueta numérica directamente en cada barra
#   - color="Media" + color_continuous_scale="Blues"
#     → gradiente de azul proporcional al valor: años más húmedos más oscuros
#   - textposition="outside" → etiquetas de texto fuera de la barra (legible)
#
# add_hline(y=media_global)
#   → línea de referencia horizontal: permite identificar visualmente
#     los años por encima (húmedos) y por debajo (secos) de la media histórica
#   → relacionado con fenómenos climáticos El Niño (años secos) / La Niña (húmedos)
#
# CRITERIO DE USO: El análisis interanual detecta heterogeneidad entre años
# que puede afectar la capacidad predictiva de un modelo entrenado en
# período específico. Si hay tendencia creciente o decreciente en la media anual,
# la serie puede necesitar una diferenciación adicional (d>1 o D=1 estacional).
# =============================================================================
# Caudal promedio anual
anual = df.groupby("Año")["Caudal"].agg(["mean", "std", "median"]).reset_index()
anual.columns = ["Año", "Media", "Desv_Std", "Mediana"]

# Media global
media_global = df["Caudal"].mean()

fig = px.bar(
    anual, x="Año", y="Media",
    error_y="Desv_Std",
    title="Caudal Promedio Anual con Desviación Estándar",
    labels={"Media": "Caudal Promedio (m³/s)", "Año": ""},
    text=anual["Media"].round(1),
    color="Media",
    color_continuous_scale="Blues",
)
fig.add_hline(y=media_global, line_dash="dash", line_color="red",
              annotation_text=f"Media global = {media_global:.1f} m³/s")
fig.update_traces(textposition="outside")
fig.update_layout(width=900, height=500, showlegend=False)
fig.show()

print("📊 Resumen anual:")
display(anual.round(2))


📊 Resumen anual:


,Año,Media,Desv_Std,Mediana
0,2010,3.58,1.71,3.27
1,2011,4.33,0.79,4.12
2,2012,3.25,0.34,3.20
3,2013,3.98,0.54,3.82
4,2014,5.27,1.80,4.77
5,2015,2.93,1.72,2.39
6,2016,1.23,0.77,1.39
7,2017,3.26,0.89,2.98


In [11]:

# =============================================================================
# CONCEPTO: Series anuales superpuestas — comparación de estacionalidad interanual
# -----------------------------------------------------------------------------
# Técnica de superposición por "día del año":
#   grupo_plot["DiaDelAnio"] = grupo_plot.index.dayofyear
#   → transforma el eje X de fechas absolutas a posición dentro del año (1–365)
#   → permite trazar todos los años sobre el mismo eje X para comparar la
#     forma de la curva estacional año a año
#
# df.groupby("Año") → itera sobre cada grupo (año, subDataFrame)
#   para i, (anio, grupo): desempaqueta el índice del grupo y el DataFrame
#
# Patrón de color por año:
#   colores_anio = px.colors.qualitative.Plotly
#   colores_anio[i % len(colores_anio)] → cicla sobre la paleta si hay más
#   años que colores disponibles (evita IndexError)
#
# go.Scatter por año → una traza por año = una línea en la leyenda
#   line=dict(width=1.2) → trazo delgado para distinguir múltiples años superpuestos
#
# Interpretación del gráfico:
#   - Si todas las líneas siguen trayectorias similares → estacionalidad estable
#   - Si una línea se aleja sistemáticamente → año atípico (El Niño, La Niña, dato inconsistente)
#   - La superposición visual complementa el gráfico de barras anuales
#
# CRITERIO DE USO: Indispensable en EDA de series hidro-meteorológicas para
# verificar que el patrón estacional es reproducible. Un año con forma muy
# distinta podría justificar excluirlo del entrenamiento o tratarlo con un
# modelo diferenciado.
# =============================================================================
# Series superpuestas por año para comparar estacionalidad interanual
fig = go.Figure()

colores_anio = px.colors.qualitative.Plotly
for i, (anio, grupo) in enumerate(df.groupby("Año")):
    # Normalizar al día del año para superponer
    grupo_plot = grupo.copy()
    grupo_plot["DiaDelAnio"] = grupo_plot.index.dayofyear
    fig.add_trace(go.Scatter(
        x=grupo_plot["DiaDelAnio"], y=grupo_plot["Caudal"],
        mode="lines", name=str(anio),
        line=dict(width=1.2, color=colores_anio[i % len(colores_anio)]),
    ))

fig.update_layout(
    title="Caudal Diario Superpuesto por Año (Comparación de Estacionalidad)",
    xaxis_title="Día del año", yaxis_title="Caudal (m³/s)",
    width=1000, height=500,
    hovermode="x unified",
    legend_title="Año",
)
fig.show()


## 10. Distribución por Épocas Hidrológicas (Seca vs Lluviosa)

En Colombia, el régimen hidrológico típico se divide en:
- **Época seca:** Dic–Ene–Feb (y Jul–Ago en bimodal)
- **Época lluviosa:** Mar–May y Sep–Nov

Clasificamos los meses en **época seca** y **época lluviosa** para comparar caudales.

In [12]:

# =============================================================================
# CONCEPTO: Clasificación de épocas hidrológicas y análisis comparativo
# -----------------------------------------------------------------------------
# Colombia tiene régimen bimodal: dos épocas lluviosas y dos épocas secas por año
#   Seca:     Dic–Ene–Feb (1er semestre) y Jul–Ago (2do semestre)
#   Lluviosa: Mar–May (1er semestre) y Sep–Nov (2do semestre)
#
# def clasificar_epoca(mes):
#   → función pura aplicada elemento a elemento con .apply()
#   → if mes in [12, 1, 2, 7, 8]: Seca; else: Lluviosa
#   → encapsula la lógica de dominio en la función evitando vectorización manual
#
# df["Mes"].apply(clasificar_epoca)
#   → aplica la función escalar a cada valor de la columna "Mes"
#   → alternativa más legible que np.where o pd.cut para lógica de dominio
#
# px.histogram(df, x="Caudal", color="Epoca", barmode="overlay", opacity=0.7)
#   - barmode="overlay" → histogramas superpuestos (vs "stack" apilados)
#   - opacity=0.7       → transparencia para ver ambas distribuciones
#   - color_discrete_map → colores semánticos: naranja=Seca, azul=Lluviosa
#   - nbins=50          → granularidad del histograma
#
# df.groupby("Epoca")["Caudal"].describe()
#   → tabla completa (count, mean, std, min, cuartiles, max) por época
#   → permite cuantificar la diferencia de caudal entre épocas
#
# CRITERIO DE USO: La separación por épocas hidrológicas permite validar que
# el régimen estacional identificado en STL y ACF es coherente con el conocimiento
# hidrológico del dominio. Épocas con distribuciones claramente diferentes justifican
# incluir variables dummy estacionales en los modelos de regresión.
# =============================================================================
# Clasificar meses en épocas hidrológicas
def clasificar_epoca(mes):
    if mes in [12, 1, 2, 7, 8]:
        return "Seca"
    else:
        return "Lluviosa"

df["Epoca"] = df["Mes"].apply(clasificar_epoca)

# Histograma comparativo por época
fig = px.histogram(
    df.reset_index(), x="Caudal", color="Epoca",
    nbins=50, barmode="overlay", opacity=0.7,
    title="Distribución del Caudal por Época Hidrológica",
    labels={"Caudal": "Caudal (m³/s)", "count": "Frecuencia"},
    color_discrete_map={"Seca": "#ff7f0e", "Lluviosa": "#1f77b4"},
)
fig.update_layout(width=900, height=450)
fig.show()

# Estadísticas por época
print("📊 Estadísticas por época hidrológica:")
epoca_stats = df.groupby("Epoca")["Caudal"].describe().round(2)
display(epoca_stats)


📊 Estadísticas por época hidrológica:


,count,mean,std,min,25%,50%,75%,max
Epoca,,,,,,,,
Lluviosa,1704.0,3.39,1.50,0.12,2.68,3.34,4.07,12.48
Seca,1218.0,3.60,1.78,0.13,2.72,3.42,4.13,9.26


In [13]:

# =============================================================================
# CONCEPTO: Violin plot por época × año y análisis de días extremos (P90)
# -----------------------------------------------------------------------------
# px.violin con facet_col="Año"
#   → crea un panel (faceta) por cada año, mostrando la distribución
#     seca/lluviosa de forma simultánea para todos los años del período
#   - facet_col_wrap=4 → máximo 4 paneles por fila antes de saltar de línea
#   - color="Epoca"    → codificación consistente seca=naranja, lluviosa=azul
#   - box=True         → boxplot incrustado en cada violín
#   - points=False     → omite puntos individuales (serie muy densa)
#
# Análisis de días extremos por época (Percentil 90):
#   p90 = df["Caudal"].quantile(0.90)
#   → umbral: el 10% de los días con mayor caudal en toda la serie histórica
#
#   (subset["Caudal"] > p90).sum()
#   → cuenta cuántos días por época superan el P90
#   → extremos/len(subset)*100 → porcentaje de días extremos en esa época
#
# Interpretación:
#   En un régimen bimodal colombiano se espera que la época lluviosa concentre
#   la mayoría de los días extremos (> P90). Si la época seca también muestra
#   un porcentaje alto, puede indicar eventos de El Niño atípicos o errores de datos.
#
# CRITERIO DE USO: El violin facetado por año permite detectar si la separación
# seca/lluviosa es estable o si cambia entre años. El análisis de P90 cuantifica
# el riesgo hidrológico por época, relevante para gestión de recursos hídricos.
# =============================================================================
# Violin plot por época y año
fig = px.violin(
    df.reset_index(), x="Epoca", y="Caudal",
    color="Epoca", box=True, points=False,
    facet_col="Año", facet_col_wrap=4,
    title="Distribución del Caudal por Época y Año",
    labels={"Caudal": "Caudal (m³/s)", "Epoca": ""},
    color_discrete_map={"Seca": "#ff7f0e", "Lluviosa": "#1f77b4"},
)
fig.update_layout(width=1000, height=600, showlegend=True)
fig.show()

# Proporción de días extremos por época (> percentil 90)
p90 = df["Caudal"].quantile(0.90)
for epoca in ["Seca", "Lluviosa"]:
    subset = df[df["Epoca"] == epoca]
    extremos = (subset["Caudal"] > p90).sum()
    print(f"  {epoca}: {extremos} días con caudal > P90 ({p90:.1f} m³/s) "
          f"→ {extremos/len(subset)*100:.1f}% de sus días")


  Seca: 154 días con caudal > P90 (5.2 m³/s) → 12.6% de sus días
  Lluviosa: 139 días con caudal > P90 (5.2 m³/s) → 8.2% de sus días


## 11. Resumen del EDA: Dashboard de Hallazgos

Consolidamos los resultados clave del EDA en una visualización de resumen.

In [14]:

# =============================================================================
# CONCEPTO: Dashboard de resumen — composición de 4 gráficos clave del EDA
# -----------------------------------------------------------------------------
# make_subplots(rows=2, cols=2) → grilla 2×2 de subgráficos
#   - subplot_titles   → etiqueta en cada panel
#   - vertical_spacing / horizontal_spacing → separación relativa entre paneles
#   - Se reutilizan objetos calculados previamente: resultado, acf_vals, meses, ic_95
#
# Panel 1 (fila 1, col 1): Serie original + Tendencia STL
#   - Capa inferior: serie completa en azul claro (width=0.5 para densidad)
#   - Capa superior: tendencia en rojo (width=2.5 resalta sobre la serie)
#   → Permite visualizar la dirección de largo plazo in contexto
#
# Panel 2 (fila 1, col 2): Componente estacional — primer ciclo completo
#   resultado.seasonal[:365] → extrae exactamente 1 año del componente periódico
#   → Muestra la forma típica del ciclo anual sin ruido ni tendencia
#
# Panel 3 (fila 2, col 1): ACF primeros 400 lags
#   acf_vals[:401] → recorte para mayor legibilidad
#   → Pico en lag 365 confirma ciclo anual; decaimiento revela autocorrelación
#   add_hline(±ic_95) → bandas de confianza reusadas del cálculo anterior
#
# Panel 4 (fila 2, col 2): Promedios mensuales (barras)
#   df.groupby("Mes")["Caudal"].mean() → caudal promedio por mes
#   px.colors.sequential.Blues[3:] → paleta secuencial de azules (omite tonos muy claros)
#
# showlegend=False → el dashboard es autónomo; las leyendas ocupan espacio valioso
#
# CRITERIO DE USO: Un dashboard de 4 paneles es el estándar de presentación en
# informes de EDA temporal. Consolida la evidencia de: (1) tendencia, (2) forma
# estacional, (3) autocorrelación y (4) variabilidad mensual en una sola figura,
# facilitando la comunicación a audiencias técnicas y no técnicas.
# =============================================================================
# Dashboard resumen: 4 gráficos clave
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Serie + Tendencia STL",
        "Componente Estacional (1 ciclo)",
        "ACF (primeros 400 lags)",
        "Caudal Promedio por Mes",
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

# 1. Serie + tendencia
fig.add_trace(go.Scatter(x=df.index, y=df["Caudal"], mode="lines",
              name="Caudal", line=dict(color="#aec7e8", width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=resultado.trend.index, y=resultado.trend, mode="lines",
              name="Tendencia", line=dict(color="#d62728", width=2.5)), row=1, col=1)

# 2. Componente estacional (primeros 365 días)
seasonal_1yr = resultado.seasonal[:365]
fig.add_trace(go.Scatter(x=list(range(365)), y=seasonal_1yr.values, mode="lines",
              name="Estacionalidad", line=dict(color="#2ca02c", width=1.5)), row=1, col=2)

# 3. ACF
acf_400 = acf_vals[:401]
fig.add_trace(go.Bar(x=list(range(401)), y=acf_400,
              marker_color="#1f77b4", name="ACF"), row=2, col=1)
fig.add_hline(y=ic_95, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-ic_95, line_dash="dash", line_color="red", row=2, col=1)

# 4. Promedios mensuales
prom_mes = df.groupby("Mes")["Caudal"].mean()
fig.add_trace(go.Bar(x=meses, y=prom_mes.values,
              marker_color=px.colors.sequential.Blues[3:], name="Promedio mensual"), row=2, col=2)

fig.update_layout(
    title="Dashboard Resumen: EDA Avanzado del Caudal – Estación Pueblo Nuevo",
    width=1100, height=700, showlegend=False,
)
fig.show()


In [15]:

# =============================================================================
# CONCEPTO: Resumen numérico consolidado del EDA avanzado
# -----------------------------------------------------------------------------
# Esta celda consolida en un solo bloque de texto todos los indicadores
# calculados a lo largo del notebook. Sirve como "ejecutivo de resumen"
# del análisis y como punto de referencia para el modelado de la Semana 4.
#
# Indicadores reportados:
#
# Estadísticas descriptivas de la serie:
#   .mean() / .median() / .std() → tendencia central y dispersión
#   CV = std/mean * 100 → Coeficiente de Variación: dispersión relativa
#     CV > 30% indica alta variabilidad (típico en series hidrológicas)
#
# Fuerza de componentes STL (calculados en celda 7):
#   ft (tendencia) y fs (estacionalidad) reusados de resultado.resid.var()
#   Valores cercanos a 1.0 → componente domina la varianza de la serie
#
# Resultados de estacionariedad (calculados en celda 12):
#   adf_p_orig  / kpss_p_orig  → pruebas sobre la serie sin transformar
#   adf_p_diff                 → prueba sobre df["Caudal_diff"] (d=1)
#   Criterios: ADF p<0.05 → estacionaria / KPSS p≥0.05 → estacionaria
#
# ACF valores clave (calculados en celda 11):
#   acf_vals[1]   → autocorrelación con el día inmediatamente anterior
#   acf_vals[365] → autocorrelación con el mismo día del año anterior
#   acf_vals[730] → ciclo bi-anual
#   pacf_vals[1]  → primera autocorrelación parcial (orden AR)
#
# CRITERIO DE USO: El resumen numérico estandariza la comunicación de
# hallazgos del EDA entre sesiones de trabajo y facilita la escritura de
# reportes técnicos. Todos los valores deben ser interpretados en conjunto
# para tomar decisiones informadas sobre el modelo SARIMA a entrenar.
# =============================================================================
# Resumen numérico del EDA
print("=" * 65)
print("📋 RESUMEN NUMÉRICO DEL EDA AVANZADO")
print("=" * 65)
print(f"🔹 Estación:     Pueblo Nuevo (IDEAM, código 21117100)")
print(f"🔹 Período:      {df.index.min().date()} → {df.index.max().date()} ({len(df)} días)")
print(f"🔹 Variable:     Caudal medio diario (m³/s)")
print(f"\n📊 Estadísticas:")
print(f"   Media:    {df['Caudal'].mean():.2f} m³/s")
print(f"   Mediana:  {df['Caudal'].median():.2f} m³/s")
print(f"   Std:      {df['Caudal'].std():.2f} m³/s")
print(f"   CV:       {df['Caudal'].std()/df['Caudal'].mean()*100:.1f}%")
print(f"\n📐 Descomposición STL:")
ft = 1 - resultado.resid.var() / (resultado.trend + resultado.resid).var()
fs = 1 - resultado.resid.var() / (resultado.seasonal + resultado.resid).var()
print(f"   Fuerza de tendencia:      {ft:.3f}")
print(f"   Fuerza de estacionalidad: {fs:.3f}")
print(f"\n🧪 Estacionariedad:")
print(f"   ADF p-valor (original):     {adf_p_orig:.6f} → {'Estacionaria' if adf_p_orig < 0.05 else 'No estacionaria'}")
print(f"   KPSS p-valor (original):    {kpss_p_orig:.4f} → {'Estacionaria' if kpss_p_orig >= 0.05 else 'No estacionaria'}")
print(f"   ADF p-valor (diferenciada): {adf_p_diff:.6f} → {'Estacionaria' if adf_p_diff < 0.05 else 'No estacionaria'}")
print(f"\n📌 Autocorrelación:")
print(f"   ACF lag 1:   {acf_vals[1]:.4f}")
print(f"   ACF lag 365: {acf_vals[365]:.4f}")
print(f"   PACF lag 1:  {pacf_vals[1]:.4f}")


📋 RESUMEN NUMÉRICO DEL EDA AVANZADO
🔹 Estación:     Pueblo Nuevo (IDEAM, código 21117100)
🔹 Período:      2010-01-01 → 2017-12-31 (2922 días)
🔹 Variable:     Caudal medio diario (m³/s)

📊 Estadísticas:
   Media:    3.48 m³/s
   Mediana:  3.37 m³/s
   Std:      1.63 m³/s
   CV:       46.7%

📐 Descomposición STL:
   Fuerza de tendencia:      0.303
   Fuerza de estacionalidad: -0.047

🧪 Estacionariedad:
   ADF p-valor (original):     0.004799 → Estacionaria
   KPSS p-valor (original):    0.0100 → No estacionaria
   ADF p-valor (diferenciada): 0.000000 → Estacionaria

📌 Autocorrelación:
   ACF lag 1:   0.9932
   ACF lag 365: 0.0398
   PACF lag 1:  0.9935


---

## 12. Conclusiones de la Semana 3

### Hallazgos Clave:

1. **Descomposición STL:** La serie presenta una **estacionalidad anual fuerte** y una tendencia suave. Los residuos muestran variabilidad moderada, lo que indica que la mayor parte de la señal está explicada por tendencia + estacionalidad.

2. **Autocorrelación:** La ACF muestra decaimiento lento y picos significativos en **lag 365** (ciclo anual). La PACF sugiere un componente autorregresivo fuerte en lags cortos (lag 1-3), útil para definir el orden de modelos ARIMA.

3. **Estacionariedad:**
   - Serie original: probable rechazo de estacionariedad por KPSS (presencia de estacionalidad).
   - Serie diferenciada (d=1): se vuelve estacionaria → necesaria **una diferenciación** para modelos ARIMA.

4. **Pregunta de investigación formulada:** Pronóstico de caudal a 30 días usando SARIMA, Holt-Winters y Prophet, evaluados con RMSE, MAE y MAPE.

5. **Estacionalidad mensual:** Se confirma un patrón estacional marcado, con meses de alto caudal y meses de estiaje, coherente con el régimen hidrológico colombiano.

6. **Variación interanual:** Algunos años presentan caudales significativamente diferentes de la media global, lo que podría estar relacionado con fenómenos climáticos (El Niño/La Niña).

7. **Épocas hidrológicas:** La clasificación seca/lluviosa muestra distribuciones claramente diferenciadas, confirmando la utilidad de incluir estacionalidad en los modelos.

### Próximos pasos (Semana 4 – Modelado Predictivo):
- Dividir datos en **train/test** (últimos 30 o 60 días como test)
- Entrenar **SARIMA** con parámetros guiados por ACF/PACF
- Entrenar **Holt-Winters** (Exponential Smoothing triple)
- Entrenar **Prophet** con estacionalidad anual
- Comparar modelos con métricas (RMSE, MAE, MAPE)
- Visualizar pronósticos vs valores reales